# Visual Health — DDR, Green à 224 px (Kaggle)

Embeddings RETFound-Green **en 224 px** (encodages de position interpolés), prétraitement
`crop-mask`, des 12 522 images gradables de **DDR**. Analyse exploratoire, plan fixé avant
ce calcul : `scripts/ddr_resolution.py`. L'avance de Green sur RETFound tient-elle à
résolution égale ? L'analyse se fait ensuite en local, sur CPU.

Le notebook vérifie d'abord que le dataset est **identique à l'archive des auteurs**.

**Avant de lancer** :
1. `Add Input` → **Your Datasets** → `marco865/ddr-grading-officiels`.
2. `Settings > Accelerator > GPU T4 x2`, `Settings > Internet > On` (pas de secret nécessaire).
3. **`Save Version > Save & Run All (Commit)`** (~15 à 25 min). Le fichier
   `visual_health_ddr_green224.zip` apparaît ensuite dans l'onglet `Output`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : activer GPU T4 x2"
!df -h /kaggle/working /kaggle/temp | tail -2


In [ ]:
import os
import subprocess

OUTPUT = "/kaggle/working"  # seul le zip y est écrit
WORK = "/kaggle/temp"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

REPO = "sumarc183-design/visual-health-platform"
REVISION = "main"  # ou un commit précis, pour rejouer un run à l'identique


def _git(*args):
    if subprocess.run(["git", *args], capture_output=True).returncode != 0:
        raise RuntimeError(f"git {args[0]} a échoué")


if not os.path.exists("visual-health-platform"):
    _git("clone", "-q", f"https://github.com/{REPO}.git", "visual-health-platform")
_git("-C", "visual-health-platform", "fetch", "-q", "origin")
_git("-C", "visual-health-platform", "reset", "-q", "--hard", "origin/main" if REVISION == "main" else REVISION)
!cd visual-health-platform && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt


## Vérification du dataset

Empreinte de l'inventaire officiel (fichiers de stades et 13 673 images), calculée sur
l'archive des auteurs : `inventory_digest` dans `src/data/ddr_dataset.py`.


In [ ]:
import glob

hits = sorted(glob.glob("/kaggle/input/**/DR_grading/train.txt", recursive=True))
if not hits:
    raise RuntimeError(f"DDR introuvable : ajouter le dataset privé en input. Inputs : {os.listdir('/kaggle/input')}")
DDR = os.path.dirname(os.path.dirname(hits[0]))
print("DDR :", DDR)

from src.data.ddr_dataset import inventory_digest

EXPECTED = "b05f6d1711402a1e1a77f5dd0e0a7f32741d6780dd604b18533c482d7611cfcd"
digest, n = inventory_digest(DDR)
print(n, "fichiers, empreinte", digest)
if digest != EXPECTED:
    raise RuntimeError("Le dataset DDR diffère de l'archive officielle : arrêt avant tout calcul")
print("Dataset identique à l'archive officielle")


In [ ]:
GREEN = "third_party/retfound_green/retfoundgreen_statedict.pth"
if not os.path.exists(GREEN):
    !mkdir -p third_party/retfound_green
    !curl -sL --fail -o {GREEN} https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth
GREEN_SHA = "431de5dbc1bebbb32f60e2c0bcf8daa4f8bcbf06f7cb1e1dc97ec589713942e1"
found = !sha256sum {GREEN}
if found[0].split()[0] != GREEN_SHA:
    raise RuntimeError(f"Poids RETFound-Green inattendus : {found[0]}")
print(GREEN, "OK")

## Embeddings

Même extraction que pour la confirmation (`scripts/extract_embeddings.py`), Green en 224 px.

In [ ]:
import time

OUT = "embeddings/ddr_crop-mask_224_retfoundgreen_statedict.npz"
t0 = time.time()
!python -u scripts/extract_embeddings.py --dataset ddr --backbone green --image-size 224 --data-root "{DDR}" \
    --checkpoint {GREEN} --out-dir embeddings --batch-size 64 --num-workers 4 2>&1 | grep -v -i warning
if not os.path.exists(OUT) or os.path.getmtime(OUT) < t0:
    raise RuntimeError("extract_embeddings.py a échoué (voir la sortie ci-dessus)")
print(f"green 224 : {(time.time() - t0) / 60:.0f} min")

## Garder le résultat


In [ ]:
import zipfile

archive = f"{OUTPUT}/visual_health_ddr_green224.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in (OUT, "embeddings/ddr_labels.csv"):
        z.write(f)
subprocess.run(["python", "scripts/run_manifest.py", "--archive", archive, "--requested-revision", REVISION], check=True)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")